In [1]:
from spotlight.datasets.goodbooks import get_goodbooks_dataset, _get_dataset
from spotlight.interactions import Interactions


In [2]:
import pandas as pd
books = pd.read_csv('../../Potomac/AIT 620/LabWorks/PythonProject/datafiles/books.csv', index_col=0)

In [3]:
books.head()

,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
book_id,,,,,,,,,,,,,,,,,,,,,
1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [4]:
def get_book_titles(book_ids):
  if isinstance(book_ids, int):
    book_ids = [book_ids]
  titles = []
  for book_id in book_ids:
    titles.append(books.loc[book_id, 'title'])
  return titles

In [5]:
data = _get_dataset()
interactions = Interactions(*data)

In [6]:
data

(array([    1,     2,     2, ..., 49925, 49925, 49925], dtype=int32),
 array([ 258, 4081,  260, ...,  722,  949, 1023], dtype=int32),
 array([5., 4., 5., ..., 4., 5., 4.], dtype=float32),
 array([      0,       1,       2, ..., 5976476, 5976477, 5976478],
       dtype=int32))

In [7]:
print(interactions)

<Interactions dataset (53425 users x 10001 items x 5976479 interactions)>


In [8]:
import torch

from spotlight.factorization.explicit import ExplicitFactorizationModel

# model = ExplicitFactorizationModel(loss='regression',
                                   embedding_dim=128,  # latent dimensionality
                                   n_iter=10,  # number of epochs of training
                                   batch_size=1024,  # minibatch size
                                   l2=1e-9,  # strength of L2 regularization
                                   learning_rate=1e-3,
                                   use_cuda=torch.cuda.is_available())

print("MODEL TRAINING COMPLETE: ", model)


MODEL TRAINING COMPLETE:  <ExplicitFactorizationModel: [uninitialised]>


In [9]:
from spotlight.cross_validation import random_train_test_split
import numpy as np

train, test = random_train_test_split(interactions, random_state=np.random.RandomState(42))


In [10]:
print('Split into \n {} and \n {}.'.format(train, test))

Split into 
 <Interactions dataset (53425 users x 10001 items x 4781183 interactions)> and 
 <Interactions dataset (53425 users x 10001 items x 1195296 interactions)>.


In [11]:
model.fit(train, verbose=True)
from spotlight.evaluation import rmse_score, precision_recall_score

train_rmse = rmse_score(model, train)
test_rmse = rmse_score(model, test)
train_precision, train_recall = precision_recall_score(model, train, k=5)
test_precision, test_recall = precision_recall_score(model, test, k=5)

print('Train RMSE {:.3f}, test RMSE {:.3f}'.format(train_rmse, test_rmse))
print(
    'mean train precision at 5: {:.3f}'.format(
        train_precision.mean()
))
print(
    'mean test precision at 5: {:.3f}'.format(
        test_precision.mean()
))

Epoch 0: loss 2.755874972080521
Epoch 1: loss 0.7533743654762652
Epoch 2: loss 0.675948987196038
Epoch 3: loss 0.5755439044193742
Epoch 4: loss 0.45039031649171923
Epoch 5: loss 0.3306060429686397
Epoch 6: loss 0.23879302652987475
Epoch 7: loss 0.17545695521208662
Epoch 8: loss 0.13363280754687953
Epoch 9: loss 0.1060949973953485
Train RMSE 0.264, test RMSE 0.963
mean train precision at 5: 0.028
mean test precision at 5: 0.019


In [12]:
# explaining predictions. Based on https://github.com/lyst/lightfm/blob/master/examples/quickstart/quickstart.ipynb

def sample_recommendation(model, user_ids, train, item_labels):
    '''Give recommendations for users given a model and explain recommendations.
    '''
    n_users, n_items = train.shape

    for user_id in user_ids:
        known_positives = item_labels[train[user_id].indices]
        
        scores = model.predict(user_id, np.arange(n_items))
        top_items = item_labels[np.argsort(-scores)]
        
        print("User %s" % user_id)
        print("     Known positives:")
        
        for x in known_positives[:3]:
            print("        %s" % x)

        print("     Recommended:")
        
        for x in top_items[:3]:
            print("        %s" % x)

In [13]:
book_labels = get_book_titles(list(train.item_ids))
book_labels[:10]

["Ahab's Wife, or The Star-Gazer",
 'City of Glass (The Mortal Instruments, #3)',
 "Enchanters' End Game (The Belgariad, #5)",
 'Frankenstein',
 'The Atlantis Complex (Artemis Fowl, #7)',
 'The Life and Times of the Thunderbolt Kid',
 'A Game of Thrones (A Song of Ice and Fire, #1)',
 'Disgrace',
 'Beautiful Creatures (Caster Chronicles, #1)',
 'The Alchemist']

In [14]:
sample_recommendation(model, [3, 9999, 15000], train.tocsr(), np.array(book_labels))

User 3
     Known positives:
        The Atlantis Complex (Artemis Fowl, #7)
        Sentinel (Covenant, #5)
        The Devil Wears Prada (The Devil Wears Prada, #1)
     Recommended:
        Beautiful Creatures (Caster Chronicles, #1)
        Palace Walk (The Cairo Trilogy #1)
        The Count of Monte Cristo
User 9999
     Known positives:
        City of Glass (The Mortal Instruments, #3)
        The Magicians' Guild (Black Magician Trilogy, #1)
        Bridge to Terabithia
     Recommended:
        Where'd You Go, Bernadette
        Delirium (Delirium, #1)
        Dreamless (Starcrossed, #2)
User 15000
     Known positives:
        Enchanters' End Game (The Belgariad, #5)
        The Life and Times of the Thunderbolt Kid
        Beautiful Creatures (Caster Chronicles, #1)
     Recommended:
        Tempted (House of Night, #6)
        Blink: The Power of Thinking Without Thinking
        The Murderer's Daughters


In [15]:
from spotlight.evaluation import precision_recall_score

train_prs = precision_recall_score(model, train, k=5)
test_prs = precision_recall_score(model, test, k=5)

print('Train PRS {:.3f}, test PRS {:.3f}'.format(train_rmse, test_rmse))

Train PRS 0.264, test PRS 0.963


In [16]:
print(
    'mean train precision at 5: {:.3f}'.format(
        train_prs[0].mean()
))
print(
    'mean test precision at 5: {:.3f}'.format(
        test_prs[0].mean()
))

mean train precision at 5: 0.028
mean test precision at 5: 0.019


In [53]:
# from tutorial at https://github.com/lyst/lightfm
# from lightfm import LightFM
# from lightfm.evaluation import precision_at_k

# Load the MovieLens 100k dataset. Only five
# star ratings are treated as positive.
#data = fetch_movielens(min_rating=5.0)

# Instantiate and train the model
# model = LightFM(loss='warp')
# model.fit(train.tocoo(), epochs=30, num_threads=2)

########## Using LightFM cannot be tested due to MS Visual update depdendency and may impact on my windows.

In [18]:
# Evaluate the trained model
# test_precision = precision_at_k(model, test.tocoo(), k=5)

NameError: name 'precision_at_k' is not defined

In [19]:
test_precision  # .mean()

array([0., 0., 0., ..., 0., 0., 0.])

In [20]:
test_precision.mean()

np.float64(0.01886863090112688)

In [21]:
sample_recommendation(model, [3, 9999, 15000], train.tocsr(), np.array(book_labels))

User 3
     Known positives:
        The Atlantis Complex (Artemis Fowl, #7)
        Sentinel (Covenant, #5)
        The Devil Wears Prada (The Devil Wears Prada, #1)
     Recommended:
        Beautiful Creatures (Caster Chronicles, #1)
        Palace Walk (The Cairo Trilogy #1)
        The Count of Monte Cristo
User 9999
     Known positives:
        City of Glass (The Mortal Instruments, #3)
        The Magicians' Guild (Black Magician Trilogy, #1)
        Bridge to Terabithia
     Recommended:
        Where'd You Go, Bernadette
        Delirium (Delirium, #1)
        Dreamless (Starcrossed, #2)
User 15000
     Known positives:
        Enchanters' End Game (The Belgariad, #5)
        The Life and Times of the Thunderbolt Kid
        Beautiful Creatures (Caster Chronicles, #1)
     Recommended:
        Tempted (House of Night, #6)
        Blink: The Power of Thinking Without Thinking
        The Murderer's Daughters
